# Context-Window Degradation — Kaggle Experiment Runner

Run cells from top to bottom. In Kaggle, open **Settings** (right-side panel), set **Accelerator = GPU**, and turn **Internet = On** so the notebook can clone GitHub and download PubMedQA/model weights. A T4 is suitable when available; the experiment must detect CUDA before the full run.

**Research integrity:** Study B contains a deliberately constructed synthetic contradiction. Its inserted result is a benchmark counterfactual, not a published biomedical finding. No results are pre-filled; output CSVs are produced only after a successful experiment.

## 0. Check Kaggle GPU and internet

Run this first. If CUDA is unavailable, stop and enable the GPU accelerator in Notebook Settings.

In [ ]:
import os, sys, platform, subprocess
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Kaggle:', os.path.exists('/kaggle'))
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        raise RuntimeError('GPU not detected. In Kaggle Settings, set Accelerator to GPU, then restart the session and rerun this cell.')
except ImportError:
    raise RuntimeError('PyTorch is not available in this Kaggle image.')

# Confirm GitHub is reachable. Internet must be enabled in Kaggle Settings.
subprocess.run(['git', '--version'], check=True)


## 1. Clone the experiment repository

This notebook uses the working feature branch containing the experiment pipeline and the latest notebook fixes. Kaggle's `/kaggle/working` directory is writable during the session.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/kaggle/working/AITR_Lab02_PSE2026002_ContextDegradation')
REPO_URL = 'https://github.com/HarshKumarSinghCU/AITR_Lab02_PSE2026002_ContextDegradation.git'
BRANCH = 'feat/reproducible-experiment-pipeline'

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Repository:', Path.cwd())
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
print('Latest commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


## 2. Install project dependencies

Kaggle images already include PyTorch and CUDA support. This cell leaves the preinstalled PyTorch untouched and installs the remaining packages declared by the project.

In [ ]:
%pip install -q 'transformers>=4.40.0' 'datasets>=2.18.0' 'sentence-transformers>=2.7.0' 'accelerate>=0.29.0' 'pandas>=2.0.0' 'matplotlib>=3.7.0' 'numpy>=1.24.0' 'PyYAML>=6.0' 'scikit-learn>=1.3.0'
print('Dependency installation finished. If pip asks for a kernel restart, restart and rerun from Step 0.')


## 3. Validate environment and run unit tests

Do not start the expensive experiment unless this cell passes.

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), 'CUDA is unavailable. Enable Kaggle GPU and restart the session.'
subprocess.run([sys.executable, 'scripts/env_sanity_check.py'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
print('Environment check and unit tests passed.')


## 4. Run the five-cluster pilot (135 experimental records)

The first run downloads the PubMedQA dataset and model weights. Keep the notebook session active. This can take time and GPU memory. If it fails, capture the **first full traceback**; do not continue to plotting.

In [ ]:
from pathlib import Path
import subprocess, sys, json

results_dir = Path('results')
results_dir.mkdir(exist_ok=True)
expected = [results_dir/'experiment_results.csv', results_dir/'summary_metrics.csv', results_dir/'run_metadata.json']
for p in expected:
    if p.exists(): p.unlink()  # prevent stale outputs being mistaken for a fresh run

subprocess.run([sys.executable, 'scripts/run_factorial_experiment.py', '--clusters', '5'], check=True)
missing = [str(p) for p in expected if not p.is_file()]
if missing:
    raise RuntimeError('Run ended without all expected outputs: ' + ', '.join(missing))
run_info = json.loads((results_dir/'run_metadata.json').read_text())
print('Run metadata:', json.dumps(run_info, indent=2))
print('Raw rows:', sum(1 for _ in (results_dir/'experiment_results.csv').open()) - 1)
print('Pilot run outputs verified.')


## 5. Generate plots and inspect measured results

This step only reads the actual CSV created in Step 4. It does not generate mock values.

In [ ]:
from pathlib import Path
import subprocess, sys, pandas as pd

raw = Path('results/experiment_results.csv')
summary_file = Path('results/summary_metrics.csv')
if not raw.is_file() or not summary_file.is_file():
    raise FileNotFoundError('Run Step 4 successfully before plotting.')
subprocess.run([sys.executable, 'scripts/plot_from_csv.py'], check=True)
summary = pd.read_csv(summary_file)
display(summary)
print('Result files:')
for p in sorted(Path('results').glob('*')):
    print(p.name, '-', p.stat().st_size, 'bytes')


## 6. Package the pilot outputs for download

After the notebook finishes, use the Kaggle **Output** panel or download this ZIP from the notebook output. Save the raw CSV, summary, run metadata, environment report, and figures.

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path('/kaggle/working/context_degradation_pilot_outputs.zip')
include = [p for p in Path('results').glob('*') if p.is_file()]
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in include:
        zf.write(p, arcname='results/' + p.name)
print('Created:', zip_path)
print('Included files:', [p.name for p in include])


## 7. Optional extended experiment (405 records)

Only run this after the pilot completes and you have enough GPU time. This backs up the pilot CSVs before running 15 clusters. It replaces the active results folder with the extended run.

In [ ]:
# OPTIONAL: remove the leading # symbols and run only when ready.
# import shutil, subprocess, sys
# from pathlib import Path
# results = Path('results')
# backup = Path('results_pilot_backup')
# backup.mkdir(exist_ok=True)
# for name in ['experiment_results.csv', 'summary_metrics.csv', 'run_metadata.json']:
#     src = results / name
#     if src.exists(): shutil.copy2(src, backup / name)
# subprocess.run([sys.executable, 'scripts/run_factorial_experiment.py', '--clusters', '15'], check=True)
# subprocess.run([sys.executable, 'scripts/plot_from_csv.py'], check=True)
# print('Extended run finished. Pilot backups are in results_pilot_backup/.')


## Troubleshooting

- **CUDA unavailable:** Kaggle Notebook Settings → Accelerator → GPU; save/restart, then rerun from Step 0.
- **Git clone/network error:** ensure Internet is On in Kaggle Settings. Kaggle may restrict Internet on some accounts/notebooks.
- **GPU quota/unavailable:** Kaggle GPU access is subject to availability and usage limits. Do not assume a reset time; retry later or use a permitted alternative runtime.
- **Out-of-memory error:** share the first traceback. Do not silently change the model or experimental settings; any protocol change must be recorded.
- **Missing CSV:** the experiment did not complete. Fix the Step 4 error first; the plotting step cannot create results.
- **Session lost:** files in `/kaggle/working` are temporary until you save/download the notebook output. Download the ZIP after a successful run.
- **Reporting:** label the five-cluster run as a pilot and the 15-cluster run as the extended experiment. Keep `run_metadata.json` and raw rows with all reported metrics.